> 🔑 **ÖĞRETMEN SÜRÜMÜ** — Bu defter, öğrenci sürümüyle aynıdır; ek olarak sonda **çözümler** bölümü vardır. Öğrencilerle bu bağlantıyı paylaşmayın.

# 🐧 LAB 2: Penguen Dedektifi — Karar Ağaçları

> **Görev:** Antarktika'daki bilim insanları üç farklı penguen türünü inceliyor. Elimizde 344 penguenin vücut ölçüleri var.
> Önce **sen** bir dedektif gibi kurallar yazacaksın, sonra **yapay zekâ** kendi kurallarını bulacak. Bakalım kim kazanacak? 🕵️ vs 🤖

![Palmer Penguenleri](https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png)
*Görsel: Allison Horst (palmerpenguins projesi)*

| Tür | Türkçe adı | Özelliği |
|-----|------------|----------|
| **Adelie** | Adelie pengueni | En küçükleri, kısa gagalı |
| **Chinstrap** | Çenebantlı penguen | Çenesinin altında siyah bir bant var |
| **Gentoo** | Papua pengueni | En iriler, turuncu gagalı |

### 🗺️ Laboratuvar Haritası
1. 🔍 Veriyi keşfet ve temizle
2. 📊 Grafiklerle ipuçlarını bul
3. 🕵️ **Kendi kurallarını yaz** ve test et
4. 🌳 Yapay zekâ kendi kurallarını öğrensin: **Karar Ağacı**
5. 🧠 Ezberleme tuzağı: Ağaç ne kadar derin olmalı?
6. 🔬 Yeni bir penguen bulduk! Türünü tahmin et

> ⏱️ Süre: yaklaşık 2 ders saati &nbsp;|&nbsp; 🎮 Hücreleri sırayla **Shift + Enter** ile çalıştır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# Her türe sabit bir renk verelim ki tüm grafiklerde aynı olsun (palette = renk paleti)
RENKLER = {"Adelie": "darkorange", "Chinstrap": "mediumorchid", "Gentoo": "teal"}
print("Hazırız! 🐧")

---
## 1️⃣ Veriyi Keşfet 🔍

Veriler 2007-2009 yılları arasında Antarktika'daki **Palmer Araştırma İstasyonu**'nda, gerçek bilim insanları tarafından toplandı.

In [ ]:
penguenler = sns.load_dataset("penguins")   # load_dataset: Seaborn'un internetteki veri setlerinden birini indir
print("Tablonun şekli:", penguenler.shape, "→ 344 penguen, 7 özellik")
penguenler.head(8)

### 📖 Sütunlar ne anlama geliyor?
| Sütun (İngilizce) | Türkçesi | Birim |
|-------------------|----------|-------|
| `species` | Tür | – |
| `island` | Yaşadığı ada (*Biscoe, Dream, Torgersen*) | – |
| `bill_length_mm` | Gaga uzunluğu | milimetre |
| `bill_depth_mm` | Gaga kalınlığı (derinliği) | milimetre |
| `flipper_length_mm` | Yüzgeç (kanat) uzunluğu | milimetre |
| `body_mass_g` | Vücut ağırlığı | gram |
| `sex` | Cinsiyet (*Male* = erkek, *Female* = dişi) | – |

![Gaga ölçüleri](https://allisonhorst.github.io/palmerpenguins/reference/figures/culmen_depth.png)

Okumayı kolaylaştırmak için sütun adlarını Türkçeye çevirelim:

In [ ]:
penguenler = penguenler.rename(columns={      # rename (yeniden adlandır)
    "species": "tur",
    "island": "ada",
    "bill_length_mm": "gaga_uzunlugu",
    "bill_depth_mm": "gaga_kalinligi",
    "flipper_length_mm": "yuzgec_uzunlugu",
    "body_mass_g": "agirlik",
    "sex": "cinsiyet",
})
penguenler.head()

### 🧹 Veri Temizliği: Eksik değerler
Gerçek hayatta veri **hiçbir zaman mükemmel değildir**. Bazı penguenler ölçüm sırasında kaçmış olabilir! 😄
Eksik değerlere **NaN** (*Not a Number* — sayı değil, yani boş) denir.

In [ ]:
print("Her sütunda kaç eksik değer var?")
print(penguenler.isnull().sum())          # isnull (boş mu?) → True/False; sum → True'ları say

In [ ]:
# Eksik satırlara bir bakalım
penguenler[penguenler.isnull().any(axis=1)]   # any(axis=1): satırdaki herhangi bir değer boşsa

In [ ]:
# Sadece 11 satır eksik. Bunları silmek en kolay ve güvenli yol.
penguenler = penguenler.dropna().reset_index(drop=True)   # dropna (drop NA = boşları at), reset_index: sıra numaralarını yeniden düzenle
print("Temizlik sonrası penguen sayısı:", len(penguenler))

In [ ]:
# Her türden kaç penguen var?
plt.figure(figsize=(7, 3.5))
sns.countplot(data=penguenler, x="tur", hue="tur", palette=RENKLER)
plt.title("Her türden kaç penguen var?")
plt.xlabel("Tür")
plt.ylabel("Penguen sayısı")
plt.show()

---
## 2️⃣ İpucu Avı: Türleri Ne Ayırıyor? 📊

Bir dedektif gibi ipucu arayalım. Hangi ölçüm, türleri birbirinden **en iyi** ayırıyor?

In [ ]:
olcumler = ["gaga_uzunlugu", "gaga_kalinligi", "yuzgec_uzunlugu", "agirlik"]
basliklar = ["Gaga uzunluğu (mm)", "Gaga kalınlığı (mm)", "Yüzgeç uzunluğu (mm)", "Ağırlık (g)"]

fig, eksenler = plt.subplots(1, 4, figsize=(17, 4))
for eksen, olcum, baslik in zip(eksenler, olcumler, basliklar):
    sns.kdeplot(data=penguenler, x=olcum, hue="tur", fill=True, palette=RENKLER,   # kdeplot: değerlerin dağılımını yumuşak tepeler şeklinde çizer
                alpha=0.4, ax=eksen, legend=(olcum == "agirlik"))
    eksen.set_title(baslik)
    eksen.set_xlabel("")
    eksen.set_ylabel("")
plt.suptitle("Her ölçümde türler nasıl dağılıyor?", fontsize=14, y=1.03)
plt.show()

> 🕵️ **İpuçlarını okuyalım:** Tepeler birbirinden ne kadar **ayrıksa**, o ölçüm türleri o kadar iyi ayırıyor.
> - **Yüzgeç uzunluğu** ve **ağırlık**: Gentoo (yeşil) diğerlerinden çok farklı! 💪 Ama Adelie ile Chinstrap üst üste biniyor.
> - **Gaga uzunluğu**: Adelie (turuncu) diğerlerinden ayrılıyor.
> - Yani **tek bir ölçüm yetmiyor**, ölçümleri **birleştirmemiz** lazım!

İki ölçümü aynı anda görmek için **dağılım grafiği** (*scatter plot*) çizelim:

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=penguenler, x="yuzgec_uzunlugu", y="gaga_uzunlugu",
                hue="tur", style="tur", palette=RENKLER, s=70)   # style: her tür için farklı nokta şekli
plt.title("Yüzgeç uzunluğu ve gaga uzunluğu", fontsize=14)
plt.xlabel("Yüzgeç uzunluğu (mm)")
plt.ylabel("Gaga uzunluğu (mm)")
plt.show()

> 🎉 **Büyük keşif!** Bu iki ölçümle üç tür neredeyse **üç ayrı küme** halinde duruyor!
> - Sol alt köşe → 🟠 Adelie (kısa yüzgeç, kısa gaga)
> - Sol üst köşe → 🟣 Chinstrap (kısa yüzgeç, uzun gaga)
> - Sağ taraf → 🟢 Gentoo (uzun yüzgeç)

Tüm ölçümlerin ikili kombinasyonlarını tek seferde görelim:

In [ ]:
# pairplot (çift grafik): tüm ölçümleri ikişer ikişer karşılaştıran grafik tablosu
sns.pairplot(penguenler, vars=olcumler, hue="tur", palette=RENKLER, height=2.3, plot_kws={"s": 20})
plt.show()

---
## 3️⃣ 🕵️ Dedektif Sensin: Kendi Kurallarını Yaz!

Yukarıdaki grafiklere bakarak, bir penguenin türünü bulan **kendi kurallarını** yazacaksın.
Aşağıdaki fonksiyonda `if`-`else` kurallarını değiştir. Sonra hücreyi çalıştır ve kurallarının **kaç penguende doğru** çıktığını gör!

> 💡 **İpucu:** Dağılım grafiğine tekrar bak. Gentoo'ları ayırmak için yüzgeç uzunluğu iyi bir başlangıç. Peki Adelie ile Chinstrap'ı ne ayırıyor?

In [ ]:
def dedektif_kurallari(penguen):
    """Bir penguenin ölçümlerine bakarak türünü tahmin eden KENDİ kurallarımız."""
    # 🎯 Bu kuralları değiştirerek daha iyi bir dedektif ol!
    if penguen["yuzgec_uzunlugu"] > 210:
        return "Gentoo"
    else:
        return "Adelie"


# Kurallarımızı tüm penguenlerde test edelim
dedektif_tahminleri = penguenler.apply(dedektif_kurallari, axis=1)   # apply (uygula): fonksiyonu her satıra uygula
dogru_sayisi = (dedektif_tahminleri == penguenler["tur"]).sum()
print(f"🕵️ Dedektif doğruluğu: %{dogru_sayisi / len(penguenler) * 100:.1f}  ({dogru_sayisi}/{len(penguenler)} penguen)")

# Hangi türlerde başarılıyız, hangilerinde değiliz? crosstab (çapraz tablo) gösterir:
pd.crosstab(penguenler["tur"], dedektif_tahminleri, rownames=["GERÇEK tür"], colnames=["DEDEKTİF tahmini"])

> 🤔 Şu anki kurallar Chinstrap türünü **hiç tanımıyor!** Çünkü hiçbir kural "Chinstrap" cevabını vermiyor.
>
> **🎯 Görev:** Kurallara bir `elif` ekleyerek Chinstrap'ı da yakalamaya çalış. **%90'ın üzerine** çıkabilir misin? Sınıfta en yüksek doğruluğu kim bulacak? 🏆
>
> Bitirince aşağıdaki bölüme geç: Bakalım yapay zekâ senden daha iyi kurallar bulabilecek mi?

---
## 4️⃣ 🌳 Yapay Zekâ Kendi Kurallarını Buluyor: Karar Ağacı

**Karar ağacı** (*decision tree*), tam olarak senin yaptığın şeyi yapar: **evet/hayır soruları** sorarak karar verir!
Farkı şu: Hangi soruyu, hangi sayıyla soracağını **veriden kendisi öğrenir.**

```
                  Yüzgeç uzunluğu ≤ 206 mu?
                   /                    \
                Evet                    Hayır
                 /                        \
     Gaga uzunluğu ≤ 43 mü?           → Gentoo 🟢
        /           \
     Evet          Hayır
      /               \
  Adelie 🟠      Chinstrap 🟣
```

**20 Soru** oyununu bilir misin? "Canlı mı?", "Uçabiliyor mu?"... Karar ağacı da en iyi soruları sırayla sorar! Her soruda, penguenleri **olabildiğince saf gruplara** (hep aynı türden oluşan gruplara) ayırmaya çalışır.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree   # DecisionTreeClassifier: karar ağacı sınıflandırıcısı, plot_tree: ağacı çiz
from sklearn.metrics import accuracy_score

X = penguenler[olcumler]      # özellikler (features): 4 vücut ölçüsü
y = penguenler["tur"]         # etiket (label): tür

X_egitim, X_test, y_egitim, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"📖 Eğitim: {len(X_egitim)} penguen  |  📝 Test: {len(X_test)} penguen")

In [ ]:
# max_depth (maksimum derinlik) = 2 → ağaç en fazla 2 soru sorabilir (tıpkı bizim kurallarımız gibi!)
agac = DecisionTreeClassifier(max_depth=2, random_state=0)
agac.fit(X_egitim, y_egitim)

dogruluk = accuracy_score(y_test, agac.predict(X_test))
print(f"🌳 Karar ağacının test doğruluğu: %{dogruluk * 100:.1f}")

### 🔎 Ağacın beynine bakalım!
Karar ağaçlarının **en güzel** yanı: Ne düşündüğünü **görebiliriz!** Birçok yapay zekâ modeli bir "kara kutu" gibidir, ama karar ağacı tamamen şeffaftır.

In [ ]:
plt.figure(figsize=(16, 7))
kutular = plot_tree(agac,
          feature_names=olcumler,               # feature names (özellik adları)
          class_names=list(agac.classes_),      # class names (sınıf adları): türler
          filled=True,                          # filled (dolu): kutuları renklendir
          rounded=True,                         # rounded (yuvarlak köşeli kutular)
          fontsize=12,
          impurity=False)                       # impurity (saflık ölçüsü) sayısını gizle, kafa karıştırmasın

# Kutuları bizim tür renklerimizle boyayalım: renk = kutudaki çoğunluk türü, koyuluk = ne kadar "saf" olduğu
from matplotlib.colors import to_rgba          # to_rgba: renk adını (kırmızı, yeşil, mavi, saydamlık) sayılarına çevirir
kutular = [k for k in kutular if "samples" in k.get_text()]   # sadece soru/karar kutularını al ("True/False" yazılarını değil)
for kutu, dagilim in zip(kutular, agac.tree_.value[:, 0, :]):    # tree_.value: her kutudaki türlerin dağılımı
    cogunluk = agac.classes_[dagilim.argmax()]
    saflik = dagilim.max() / dagilim.sum()                         # 1 = hepsi aynı tür, 0.33 = karmakarışık
    kutu.set_bbox(dict(boxstyle="round", facecolor=to_rgba(RENKLER[cogunluk], alpha=0.15 + 0.6 * saflik), edgecolor="gray"))   # bbox: yazının etrafındaki kutu
plt.title("Yapay zekânın öğrendiği kurallar", fontsize=16)
plt.show()

> 📖 **Ağaç nasıl okunur?**
> - En üstteki kutu **kök** (*root*): İlk soru burada sorulur. Soru **doğruysa** sola, **yanlışsa** sağa gidilir.
> - `samples` (örnekler): O kutuya kaç penguen ulaştı.
> - `value` (değer): O kutudaki penguenlerin türlere göre dağılımı `[Adelie, Chinstrap, Gentoo]`.
> - `class` (sınıf): O kutuya ulaşan penguene verilecek cevap.
> - En alttaki kutulara **yaprak** (*leaf*) denir: Kararın verildiği yer. 🍃
>
> 🤔 **Karşılaştır:** Yapay zekânın bulduğu kurallar seninkilere benziyor mu? Hangi ölçümü ve hangi sayıları seçti?

### 🗺️ Ağacın dünyayı nasıl böldüğünü görelim
İki ölçüm kullanan bir ağacın, grafiği nasıl **dikdörtgen bölgelere** ayırdığına bakalım:

In [ ]:
from matplotlib.colors import ListedColormap   # kendi renk listemizden renk haritası (colormap) yapmak için

def karar_bolgelerini_ciz(model, x_sutun, y_sutun, baslik, eksen):
    """Bir modelin iki özellik üzerindeki karar bölgelerini çizer."""
    x_min, x_max = penguenler[x_sutun].min() - 3, penguenler[x_sutun].max() + 3
    y_min, y_max = penguenler[y_sutun].min() - 2, penguenler[y_sutun].max() + 2
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))   # meshgrid: grafik alanını ızgara noktalarıyla doldur
    izgara = pd.DataFrame({x_sutun: xx.ravel(), y_sutun: yy.ravel()})
    turler = list(RENKLER.keys())
    tahmin = pd.Series(model.predict(izgara)).map({t: i for i, t in enumerate(turler)}).values.reshape(xx.shape)  # türleri 0,1,2 sayılarına çevir

    eksen.contourf(xx, yy, tahmin, alpha=0.25, cmap=ListedColormap(list(RENKLER.values())), levels=[-0.5, 0.5, 1.5, 2.5])
    sns.scatterplot(data=penguenler, x=x_sutun, y=y_sutun, hue="tur", palette=RENKLER, s=35, ax=eksen, legend=False, edgecolor="k", linewidth=0.3)
    eksen.set_title(baslik)


iki_ozellik = ["yuzgec_uzunlugu", "gaga_uzunlugu"]
fig, eksenler = plt.subplots(1, 3, figsize=(18, 5.5))
for eksen, derinlik in zip(eksenler, [1, 2, 8]):
    model = DecisionTreeClassifier(max_depth=derinlik, random_state=0).fit(penguenler[iki_ozellik], penguenler["tur"])
    karar_bolgelerini_ciz(model, *iki_ozellik, f"Derinlik = {derinlik}  ({derinlik} soru)", eksen)   # *liste: listeyi parametrelere aç
plt.suptitle("Ağaç derinleştikçe bölgeler nasıl değişiyor?", fontsize=15, y=1.02)
plt.show()

> 🔎 **Gözlem:**
> - **Derinlik 1:** Tek bir soru sorabiliyor, bu yüzden sadece 2 bölge var. Bir türü tamamen kaçırıyor!
> - **Derinlik 2:** Üç tür de güzelce ayrılıyor. 👏
> - **Derinlik 8:** Bölgeler parça parça oldu! Bazı minicik bölgeler **tek bir penguen** için oluşturulmuş... Model, kurallara uymayan her penguen için özel bir istisna yazmış. Bu iyi bir şey mi? 🤔

---
## 5️⃣ 🧠 Ezberleme Tuzağı (*Overfitting*)

Bir öğrenci düşün: Matematik sınavına çalışırken soruların **cevaplarını ezberliyor**, konuyu anlamıyor.
- Çalıştığı sorularda → 100 puan! 😎
- Sınavda yeni sorular gelince → 40 puan 😱

Yapay zekâ da aynı hatayı yapabilir! Buna **aşırı öğrenme / ezberleme** (*overfitting*) denir.
Ağaç çok derin olursa, eğitim verisindeki **her penguen için özel kural** yazar ve yeni penguenlerde şaşırır.

Hadi deney yapalım: Farklı derinliklerde ağaçlar kuralım ve hem **eğitim** hem **test** başarılarını karşılaştıralım.
Ezberlemeyi daha net görmek için bu sefer modele **sadece 2 ölçüm** verelim ve verinin **sadece %30'unu** eğitime ayıralım (az örnekle öğrenen model daha kolay ezberler).

In [ ]:
X2 = penguenler[["gaga_kalinligi", "agirlik"]]   # türleri ayırması daha zor iki ölçüm seçelim
X2_egitim, X2_test, y2_egitim, y2_test = train_test_split(X2, y, train_size=0.3, random_state=7, stratify=y)

derinlikler = range(1, 13)
egitim_basarisi, test_basarisi = [], []

for d in derinlikler:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X2_egitim, y2_egitim)
    egitim_basarisi.append(accuracy_score(y2_egitim, m.predict(X2_egitim)) * 100)   # ezberlediği sorular
    test_basarisi.append(accuracy_score(y2_test, m.predict(X2_test)) * 100)         # yeni sorular

en_iyi = derinlikler[int(np.argmax(test_basarisi))]

plt.figure(figsize=(10, 4.5))
plt.plot(derinlikler, egitim_basarisi, marker="o", linewidth=2, label="Eğitim başarısı (çalıştığı sorular)")
plt.plot(derinlikler, test_basarisi, marker="s", linewidth=2, label="Test başarısı (yeni sorular)")
plt.axvline(en_iyi, color="green", linestyle="--", alpha=0.7)
plt.text(en_iyi + 0.15, min(test_basarisi) + 1, f"En iyi derinlik: {en_iyi}", color="green")
plt.fill_between(derinlikler, egitim_basarisi, test_basarisi, alpha=0.1, color="red")   # fill_between: iki çizgi arasını boya
plt.xlabel("Ağacın derinliği (sorulan soru sayısı)")
plt.ylabel("Doğruluk (%)")
plt.title("Ezberleme Tuzağı: Daha derin ağaç her zaman daha iyi değildir!")
plt.xticks(derinlikler)
plt.legend()
plt.show()

> 📉 **Grafiği yorumlayalım:**
> - **Mavi çizgi (eğitim)** derinlik arttıkça sürekli yükselip **%100'e** ulaşıyor. Model eğitim verisini tamamen ezberledi!
> - **Turuncu çizgi (test)** bir noktadan sonra **yükselmiyor, hatta düşüyor.** Ezberlemek, yeni penguenleri tanımaya yardım etmiyor.
> - Aradaki **kırmızı boşluk**, ezberlemenin göstergesi. Boşluk büyüdükçe model "ezberci" oluyor.
>
> 🎓 **Altın kural:** Bir modeli asla **eğitim verisindeki** başarısıyla değerlendirme. Her zaman **hiç görmediği** test verisine bak!

### 📊 Hangi ölçüm daha önemli?
Karar ağacı, kararlarında **hangi ölçümleri** ne kadar kullandığını da söyleyebilir. Buna **özellik önemi** (*feature importance*) denir.

In [ ]:
# Tüm ölçümleri kullanan, makul derinlikte bir ağaç kuralım
agac = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_egitim, y_egitim)
print(f"🌳 Derinlik 3 ağacın test doğruluğu: %{accuracy_score(y_test, agac.predict(X_test)) * 100:.1f}")

onem = pd.Series(agac.feature_importances_, index=basliklar).sort_values()   # feature_importances_: her özelliğin önem puanı (toplamı 1)

plt.figure(figsize=(8, 3.5))
onem.plot.barh(color="teal")
plt.xlabel("Önem puanı (toplamı 1)")
plt.title("Karar ağacı en çok hangi ölçüme güveniyor?")
plt.show()

### 🧩 Karışıklık matrisi: Hangi türler karışıyor?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay   # karışıklık matrisini (confusion matrix) hazır çizen araç

ConfusionMatrixDisplay.from_estimator(agac, X_test, y_test, cmap="Greens", colorbar=False)   # estimator (tahminci): eğitilmiş model
plt.title("Karar Ağacı — Karışıklık Matrisi")
plt.xlabel("Modelin TAHMİNİ")
plt.ylabel("GERÇEK tür")
plt.grid(False)
plt.show()

### 🌲🌲🌲 Bonus: Bir ağaç yerine bir ORMAN!
Tek bir ağaç bazen hata yapar. Peki **yüz tane** farklı ağaç kurup **oylamaya** sunsak? Buna **Rastgele Orman** (*Random Forest*) denir.
Her ağaç verinin farklı bir parçasını görür ve farklı sorular sorar. Sonunda **çoğunluğun kararı** geçerli olur. Bir kişiye değil, bir sınıfa danışmak gibi! 🗳️

In [ ]:
from sklearn.ensemble import RandomForestClassifier

orman = RandomForestClassifier(n_estimators=200, random_state=0)   # n_estimators: ormandaki ağaç sayısı
orman.fit(X_egitim, y_egitim)

print(f"🌳 Tek ağaç (derinlik 3) : %{accuracy_score(y_test, agac.predict(X_test)) * 100:.1f}")
print(f"🌲 Rastgele Orman (200)  : %{accuracy_score(y_test, orman.predict(X_test)) * 100:.1f}")
print(f"🕵️ Senin kuralların      : %{(penguenler.loc[X_test.index].apply(dedektif_kurallari, axis=1) == y_test).mean() * 100:.1f}  (aynı test penguenleri üzerinde)")

---
## 6️⃣ 🔬 Yeni Bir Penguen Bulduk!

Bir araştırma gezisinde, daha önce hiç ölçülmemiş bir penguenle karşılaştın! 🐧 Ölçümlerini aldın.
Hangi türden olduğunu yapay zekâya soralım.

> 🎚️ **Google Colab'da** sağ tarafta **kaydırma çubukları** (*slider*) göreceksin. Değerleri değiştirince hücre otomatik çalışır!
> (Colab dışında kullanıyorsan sayıları elle değiştirip hücreyi tekrar çalıştır.)

In [ ]:
# @title 🐧 Yeni penguenin ölçümleri { run: "auto" }
gaga_uzunlugu = 48.5   # @param {type:"slider", min:30, max:60, step:0.5}
gaga_kalinligi = 15.5  # @param {type:"slider", min:13, max:22, step:0.1}
yuzgec_uzunlugu = 215  # @param {type:"slider", min:170, max:235, step:1}
agirlik = 5100         # @param {type:"slider", min:2500, max:6500, step:50}

yeni_penguen = pd.DataFrame([[gaga_uzunlugu, gaga_kalinligi, yuzgec_uzunlugu, agirlik]], columns=olcumler)

olasiliklar = orman.predict_proba(yeni_penguen)[0]     # predict_proba: her tür için ağaçların oy oranı (olasılık)
tahmin = orman.classes_[np.argmax(olasiliklar)]        # classes_: modelin bildiği türler

fig, eksen = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.3, 1]})

# Sol: Yeni penguen, diğer penguenlerin arasında nerede?
sns.scatterplot(data=penguenler, x="yuzgec_uzunlugu", y="gaga_uzunlugu", hue="tur", palette=RENKLER, s=40, alpha=0.6, ax=eksen[0])
eksen[0].scatter(yuzgec_uzunlugu, gaga_uzunlugu, marker="*", s=700, c="red", edgecolor="black", zorder=5, label="YENİ PENGUEN")  # zorder: en üstte çizilsin
eksen[0].legend(loc="lower right")
eksen[0].set_title("Yeni penguen nerede?")

# Sağ: Ormandaki ağaçların oyları
cubuklar = eksen[1].bar(orman.classes_, olasiliklar * 100, color=[RENKLER[t] for t in orman.classes_])
eksen[1].bar_label(cubuklar, fmt="%.0f%%")
eksen[1].set_ylim(0, 110)
eksen[1].set_ylabel("Ağaçların oyu (%)")
eksen[1].set_title(f"Tahmin: {tahmin}")
plt.tight_layout()
plt.show()

> 🧪 **Deney yap:** Kaydırma çubuklarıyla oynayarak:
> 1. Modelin **kararsız kaldığı** (oyların bölündüğü) bir penguen bulabilir misin? Bu penguen grafikte nerede duruyor?
> 2. Bir Adelie'yi yavaş yavaş Chinstrap'a dönüştürmek için **hangi ölçümü** değiştirmen gerekiyor?
> 3. Gerçekte var olamayacak bir penguen yap (örneğin 60 mm gaga + 2500 g ağırlık). Model ne diyor? **Model "bilmiyorum" diyebiliyor mu?** 🤔

---
## 🏆 Meydan Okumalar (Challenge)

1. **🕵️ Dedektif Rövanşı:** Kendi kurallarınla karar ağacının (derinlik 2) doğruluğunu geçebilir misin?
2. **🏝️ Ada İpucu:** Veri setinde `ada` bilgisi de var! Gentoo pengueni hangi adalarda yaşıyor? `pd.crosstab(penguenler["tur"], penguenler["ada"])` ile incele. Bu bilgi kurallarına yardımcı olur mu?
3. **🔬 Az Bilgiyle Tahmin:** Modeli sadece **tek bir ölçümle** eğit (örneğin sadece `agirlik`). Doğruluk ne kadar düşüyor? Hangi tek ölçüm en iyisi?
4. **🌳 Ağaç Mimarı:** `plot_tree` ile `max_depth=4` olan ağacı çiz. En alttaki kutularda kaç penguen kalıyor? Bu, ezberleme hakkında ne anlatıyor?

In [ ]:
# Meydan okumalar için kodunu buraya yaz 👇

---
## 🤔 Düşünelim: Model bilmediğini bilir mi?
Bu veri, Antarktika'daki bir araştırma istasyonunda **üç tür** penguenden toplandı. Modelimiz dünyada sadece bu üç tür olduğunu sanıyor!

- Modele bir **imparator penguenin** ölçülerini versen ne der? "Bilmiyorum" diyebilir mi, yoksa yine üç türden birini mi seçer?
- Bir hastanede, sadece **o hastanenin** hastalarıyla eğitilmiş bir model başka bir şehirde kullanılsa ne olabilir?
- Modelin %95 doğru olması, her **tek bir** tahminine güvenebileceğimiz anlamına gelir mi?

> 🧭 **Önemli fikir:** Model, eğitim verisinin dışındaki dünyayı tanımaz ve çoğu zaman bilmediği bir şeyle karşılaştığını **fark etmez**. Bu yüzden önemli kararlarda yapay zekânın cevabını bir **insanın** kontrol etmesi gerekir.

---
## 📝 Bugün Neler Öğrendik?
- 🧹 Gerçek veride **eksik değerler** olur. Onları temizlemek gerekir.
- 📊 Grafikler, hangi özelliklerin işe yarayacağına dair **ipuçları** verir.
- 🌳 **Karar ağacı**, evet/hayır soruları sorarak karar verir ve kurallarını **görebiliriz.**
- 🧠 Çok derin ağaçlar **ezberler** (*overfitting*). Test başarısı, eğitim başarısından daha önemlidir!
- 📏 **Özellik önemi**, modelin hangi bilgiye güvendiğini gösterir.
- 🌲 **Rastgele Orman**, birçok ağacın oylamasıyla daha sağlam kararlar verir.

🎉 **Tebrikler, artık bir yapay zekâ dedektifisin!** 🐧🕵️

---
# 🔑 Çözüm Örneği (Öğretmen İçin)

In [ ]:
def dedektif_kurallari_cozum(penguen):
    if penguen["yuzgec_uzunlugu"] > 206:
        return "Gentoo"
    elif penguen["gaga_uzunlugu"] > 43.5:
        return "Chinstrap"
    else:
        return "Adelie"

tahminler = penguenler.apply(dedektif_kurallari_cozum, axis=1)
print(f"Çözüm kurallarının doğruluğu: %{(tahminler == penguenler['tur']).mean() * 100:.1f}")

# Meydan okuma 2: Ada ipucu
print(pd.crosstab(penguenler["tur"], penguenler["ada"]))

# Meydan okuma 3: Tek ölçümle tahmin
for olcum in olcumler:
    m = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_egitim[[olcum]], y_egitim)
    print(f"Sadece {olcum:<16}: %{accuracy_score(y_test, m.predict(X_test[[olcum]])) * 100:.1f}")

---
## ✅ Defteri bitirdin mi?
Aşağıdaki hücreyi çalıştır. Önemli bölümlerin hepsini çalıştırdıysan sana bir **tamamlama kodu** verecek. Bu kodu sitede bu defterin **Tamamladım** kısmına yazınca defter tamamlanmış sayılır.

In [ ]:
# ✅ TAMAMLAMA KODU
# Bu hücre, defterin önemli bölümlerini çalıştırıp çalıştırmadığını kontrol eder.
# Hepsini çalıştırdıysan sana bir kod verir. Bu kodu Python Atölyesi sitesinde
# defterin "Tamamladım" kısmına yaz.
import hashlib
gerekenler = {
    "penguenler": "1. Veriyi Keşfet",
    "dedektif_kurallari": "3. Dedektif Sensin: Kendi Kurallarını Yaz",
    "agac": "4. Yapay Zekâ Kendi Kurallarını Buluyor: Karar Ağacı",
    "orman": "5. Ezberleme Tuzağı (Overfitting)",
    "yeni_penguen": "6. Yeni Bir Penguen Bulduk"
}
eksikler = [bolum for ad, bolum in gerekenler.items() if ad not in globals()]
if eksikler:
    print("Henüz bitmedi! Şu bölümlerin hücrelerini çalıştırmadın:")
    for bolum in dict.fromkeys(eksikler):
        print("  •", bolum)
else:
    ozet = hashlib.sha256(("atolye|" + "|".join(gerekenler)).encode()).hexdigest()
    print("🎉 Tebrikler, defteri bitirdin! Tamamlama kodun:", ozet[:6].upper())

---
## 🔙 Python Atölyesi sitesine dön
Sitedeki **5. adım: Lab'lar** sayfasında bu laboratuvarı **Tamamladım** olarak işaretle.
Üç laboratuvarın hepsini bitirince **aşama kontrolünü** ve kursun **son testini** çözeceksin. Son testin sonunda, kursun başındaki ön testle karşılaştırılmış **gelişim raporunu** göreceksin. 📈